In [1]:
import os
import glob
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import mlflow.pytorch
from torch.utils.data import Dataset, Subset
import random

import dagshub
import mlflow

# --- 1. MODEL & DATASET DEFINITIONS (Needed for a standalone file) ---

class KinectFolderDataset(Dataset):
    def __init__(self, folder_path, seq_length=30):
        self.X_seqs = []
        self.y_seqs = []
        file_paths = glob.glob(os.path.join(folder_path, "*.csv"))
        for path in file_paths:
            df = pd.read_csv(path)
            all_cols = [c for c in df.columns if c != 'FrameNo']
            x_y_cols = sorted([c for c in all_cols if c.endswith('_x') or c.endswith('_y')])
            z_cols = sorted([c for c in all_cols if c.endswith('_z')])
            x_data = df[x_y_cols].values.astype(np.float32)
            y_data = df[z_cols].values.astype(np.float32)
            if len(df) >= seq_length:
                for i in range(len(df) - seq_length + 1):
                    self.X_seqs.append(x_data[i : i + seq_length])
                    self.y_seqs.append(y_data[i : i + seq_length])
            
    def __len__(self): return len(self.X_seqs)
    def __getitem__(self, idx):
        return torch.tensor(self.X_seqs[idx]), torch.tensor(self.y_seqs[idx])

# --- 2. SETUP & LOAD MODEL ---


# This is the "Bridge" to DagsHub
dagshub.init(repo_owner="SamuelFredricBerg", repo_name="4dt907", mlflow=True)

project_name = "Project_Model_Z_Predictor_V2"
model_uri = f"models:/{project_name}@prod"

print(f"Loading model from DagsHub: {model_uri}")
try:
    loaded_model = mlflow.pytorch.load_model(model_uri)
    loaded_model.eval()
    print("✅ Model loaded successfully!")
except Exception as e:
    print(f"❌ Error: Could not find model. Have you registered it yet?\nDetails: {e}")

print(f"Loading model: {model_uri}")
loaded_model = mlflow.pytorch.load_model(model_uri)
loaded_model.eval()

# Names of the 13 joints in the correct order
joint_names = [
    "nose", "left_shoulder", "left_elbow", "right_shoulder", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip", "left_knee", 
    "right_knee", "left_ankle", "right_ankle"
]

# --- 3. PICK ONE TOTALLY RANDOM SEQUENCE ---
folder = "../../data/kinect_good_preprocessed_A9_mediapipe"

dataset = KinectFolderDataset(folder)
random_idx = random.randint(0, len(dataset) - 1)
single_input, single_target = dataset[random_idx]

# Reshape for model [Batch=1, Seq=30, Features=26]
single_input_batch = single_input.unsqueeze(0)

# --- 4. PREDICT & COMPARE ---

with torch.no_grad():
    prediction = loaded_model(single_input_batch)

# We'll look at the middle of the 1-second sequence (Frame 15)
frame_idx = 15 
pred_frame = prediction[0, frame_idx, :].numpy()
true_frame = single_target[frame_idx, :].numpy()

print(f"\n{'='*70}")
print(f" SANITY CHECK: Random Sequence Index {random_idx} (Frame {frame_idx})")
print(f"{'='*70}")
print(f"{'Joint Name':<18} | {'Pred Z (m)':<12} | {'True Z (m)':<12} | {'Diff (cm)':<10}")
print("-" * 70)

total_error = 0
for i, name in enumerate(joint_names):
    p_z = pred_frame[i]
    t_z = true_frame[i]
    diff_cm = abs(p_z - t_z) * 100
    total_error += diff_cm
    
    print(f"{name:<18} | {p_z:10.4f}   | {t_z:10.4f}   | {diff_cm:8.2f} cm")

print("-" * 70)
print(f"Average Frame Error: {total_error / len(joint_names):.2f} cm")
print(f"{'='*70}\n")

Accessing as Soppa22

Initialized MLflow to track repo "SamuelFredricBerg/4dt907"

Repository SamuelFredricBerg/4dt907 initialized!

Loading model from DagsHub: models:/Project_Model_Z_Predictor_V2@prod


✅ Model loaded successfully!
Loading model: models:/Project_Model_Z_Predictor_V2@prod



 SANITY CHECK: Random Sequence Index 2349 (Frame 15)
Joint Name         | Pred Z (m)   | True Z (m)   | Diff (cm) 
----------------------------------------------------------------------
nose               |    -0.1378   |    -0.1304   |     0.74 cm
left_shoulder      |    -0.0411   |    -0.0559   |     1.48 cm
left_elbow         |     0.0013   |    -0.0034   |     0.47 cm
right_shoulder     |    -0.2020   |    -0.1965   |     0.54 cm
right_elbow        |     0.0057   |    -0.0032   |     0.89 cm
left_wrist         |    -0.0981   |    -0.1204   |     2.23 cm
right_wrist        |     0.0107   |    -0.0055   |     1.63 cm
left_hip           |    -0.1098   |    -0.1041   |     0.57 cm
right_hip          |     0.0005   |    -0.0140   |     1.45 cm
left_knee          |    -0.0013   |     0.0034   |     0.48 cm
right_knee         |    -0.1703   |    -0.1639   |     0.63 cm
left_ankle         |     0.0198   |     0.0112   |     0.87 cm
right_ankle        |    -0.0424   |    -0.0665   |     2.